In [1]:
import numpy as np
import polars as pl
from datetime import datetime
from validation_service_c1.corrupted_measurements import detect_corrupted
from sklearn.covariance import LedoitWolf

SENSORS = ["temperature", "vibration", "pressure", "rpm", "power"]
TEST_START = datetime(2014, 1, 14)

df = pl.read_parquet(
    "../predictive_maintenance_data.parquet",
    columns=["timestamp", "machine_id", "operating_mode", "temperature",
             "vibration", "pressure", "rpm", "power", "concept_drift_active"],
)

start = datetime(2014, 1, 14, 0, 0)
end = datetime(2015, 1, 12, 23, 0)
print(df.height)

# drop year 3
year_1_and_2 = df.filter(~pl.col("timestamp").is_between(start, end))
print(year_1_and_2.height)

flag_cols = [f"{s}_{kind}" for s in SENSORS for kind in ("frozen", "spike")]
cleaned_df = (
    detect_corrupted(year_1_and_2)
      .filter(~pl.any_horizontal(flag_cols))  # drop rows where any flag is true
      .drop_nulls(SENSORS)                     # drop rows with missing values
      .drop(flag_cols)
      .with_columns(
          pl.col("operating_mode").cast(pl.Utf8),
          pl.when(pl.col("concept_drift_active"))
            .then(pl.lit("drift_1")).otherwise(pl.lit("no_drift"))
            .alias("group"),
      )
)
print(cleaned_df.height)
cleaned_df.head()


525600
350880
348749


timestamp,machine_id,operating_mode,temperature,vibration,pressure,rpm,power,concept_drift_active,group
datetime[μs],cat,str,f64,f64,f64,f64,f64,bool,str
2012-01-14 00:00:00,"""M001""","""HIGH_LOAD""",46.07526,1.64684,98.800597,2188.426973,316.167708,false,"""no_drift"""
2012-01-14 00:00:00,"""M002""","""NORMAL""",46.584259,4.618663,96.40474,1519.512205,220.213104,false,"""no_drift"""
2012-01-14 00:00:00,"""M003""","""IDLE""",40.546451,3.090226,96.318568,492.872492,59.735059,false,"""no_drift"""
2012-01-14 00:00:00,"""M004""","""NORMAL""",48.470209,2.953438,97.447963,1515.48815,211.998125,false,"""no_drift"""
2012-01-14 00:00:00,"""M005""","""NORMAL""",39.141233,1.222667,99.561297,1504.955537,209.234533,false,"""no_drift"""


In [2]:
from datetime import timedelta

drift_start = cleaned_df.filter(pl.col("concept_drift_active"))["timestamp"].min()
print("drift starts:", drift_start)

NO_DRIFT_VALID = drift_start - timedelta(days=90)  # last 3 months before drift
DRIFT_VALID = datetime(2013, 11, 1)                # last ~2.5 months of drift

is_valid = (
    ((pl.col("group") == "no_drift") & (pl.col("timestamp") >= NO_DRIFT_VALID))
    | ((pl.col("group") == "drift_1") & (pl.col("timestamp") >= DRIFT_VALID))
)
train = cleaned_df.filter(~is_valid)
valid = cleaned_df.filter(is_valid)

drift starts: 2013-07-15 00:00:00


In [3]:
from sklearn.mixture import GaussianMixture

def fit(frame, n_components=1, quantile=0.001):
    model = {}
    for (mode, group), g in frame.group_by(["operating_mode", "group"]):
        X = g.select(SENSORS).to_numpy()
        gm = GaussianMixture(n_components=n_components, covariance_type="full",
                             random_state=42).fit(X)
        ll = gm.score_samples(X)                      # log-likelihood per row
        model[(mode, group)] = {
            "gmm": gm,
            "threshold": np.quantile(ll, quantile),   # 99.9% of training rows are above this
            "n": len(X),
        }
    return model

model = fit(train)

In [4]:

def predict(model, frame):
    parts = []
    for mode in frame["operating_mode"].unique():
        part = frame.filter(pl.col("operating_mode") == mode)
        X = part.select(SENSORS).to_numpy()
        groups = [(g, p) for (m, g), p in model.items() if m == mode]
        LL = np.column_stack([p["gmm"].score_samples(X) for _, p in groups])
        T = np.array([p["threshold"] for _, p in groups])
        names = np.array([g for g, _ in groups])
        label = np.where((LL >= T).any(axis=1), names[LL.argmax(axis=1)], "outside")
        parts.append(part.with_columns(pl.Series("pred", label)))
    return pl.concat(parts)

res = predict(model, valid)
with pl.Config(tbl_rows=-1):
    print(res.group_by("operating_mode", "group", "pred").len()
             .sort("operating_mode", "group", "pred"))
print("new_drift rate:", (res["pred"] == "new_drift").mean())

shape: (20, 4)
┌────────────────┬──────────┬──────────┬───────┐
│ operating_mode ┆ group    ┆ pred     ┆ len   │
│ ---            ┆ ---      ┆ ---      ┆ ---   │
│ str            ┆ str      ┆ str      ┆ u32   │
╞════════════════╪══════════╪══════════╪═══════╡
│ HIGH_LOAD      ┆ drift_1  ┆ drift_1  ┆ 8709  │
│ HIGH_LOAD      ┆ drift_1  ┆ outside  ┆ 3     │
│ HIGH_LOAD      ┆ no_drift ┆ no_drift ┆ 10774 │
│ HIGH_LOAD      ┆ no_drift ┆ outside  ┆ 11    │
│ IDLE           ┆ drift_1  ┆ drift_1  ┆ 3154  │
│ IDLE           ┆ drift_1  ┆ no_drift ┆ 395   │
│ IDLE           ┆ drift_1  ┆ outside  ┆ 1     │
│ IDLE           ┆ no_drift ┆ drift_1  ┆ 504   │
│ IDLE           ┆ no_drift ┆ no_drift ┆ 3717  │
│ IDLE           ┆ no_drift ┆ outside  ┆ 2     │
│ NORMAL         ┆ drift_1  ┆ drift_1  ┆ 21210 │
│ NORMAL         ┆ drift_1  ┆ no_drift ┆ 18    │
│ NORMAL         ┆ drift_1  ┆ outside  ┆ 12    │
│ NORMAL         ┆ no_drift ┆ drift_1  ┆ 27    │
│ NORMAL         ┆ no_drift ┆ no_drift ┆ 25675 │
│ NOR

In [5]:
import joblib

final = fit(cleaned_df)
joblib.dump(
    {"version": "drift-classifier-v1", "sensors": SENSORS,
     "trained_until": TEST_START.isoformat(), "groups": final},
    "models/drift-classifier-v1.joblib",
)

['models/drift-classifier-v1.joblib']

In [6]:
NEW_DRIFT_START = datetime(2014, 8, 2)  # second drift, found in earlier exploration
WINDOW = 24                              # last 24 hourly rows per machine
NEW_DRIFT_SHARE = 0.7                      # share of new_drift rows that signals a new drift


def apply_recent_rows_rule(frame, window=WINDOW, new_drift_share=NEW_DRIFT_SHARE):
    """Turn raw per-row predictions into final labels using each machine's recent rows."""
    def recent_share(condition):
        return (condition.cast(pl.Float64)
                .rolling_mean(window_size=window, min_samples=1)
                .over("machine_id"))

    return (
        frame.sort("machine_id", "timestamp")
        .with_columns(
            recent_share(pl.col("pred") == "outside").alias("share_outside"),
            recent_share(pl.col("pred") == "drift_1").alias("share_drift_1"),
            recent_share(pl.col("pred") == "no_drift").alias("share_no_drift"),
        )
        .with_columns(
            # Known group: take the majority of recent rows (smooths IDLE)
            pl.when(pl.col("pred") != "outside")
              .then(pl.when(pl.col("share_drift_1") >= pl.col("share_no_drift"))
                      .then(pl.lit("drift_1")).otherwise(pl.lit("no_drift")))
            # new_drift, and recent rows are mostly new_drift too: possible new drift
              .when(pl.col("share_outside") >= new_drift_share).then(pl.lit("new_drift"))
            # new_drift, but recent rows are normal: one-off
              .otherwise(pl.lit("outlier"))
              .alias("label")
        )
    )

In [7]:
year3 = df.filter(pl.col("timestamp") >= TEST_START).sort("machine_id", "timestamp")
flagged3 = detect_corrupted(year3).with_columns(pl.col("operating_mode").cast(pl.Utf8))

has_problem = pl.any_horizontal(flag_cols) | pl.any_horizontal(pl.col(SENSORS).is_null())

classified = apply_recent_rows_rule(
    predict(final, flagged3.filter(~has_problem).drop(flag_cols))
)
skipped = (flagged3.filter(has_problem).drop(flag_cols)
           .with_columns(pl.lit(None, pl.Utf8).alias("pred"),
                         pl.lit(None, pl.Utf8).alias("label")))

result = (
    pl.concat([classified, skipped], how="diagonal")
      .sort("machine_id", "timestamp")
      .with_columns(
          pl.when(~pl.col("concept_drift_active")).then(pl.lit("no_drift"))
            .when(pl.col("timestamp") < NEW_DRIFT_START).then(pl.lit("drift_1"))
            .otherwise(pl.lit("new_drift"))
            .alias("truth")
      )
)
print(f"{result.height} rows, {skipped.height} skipped for quality problems")

174720 rows, 1077 skipped for quality problems


In [8]:
with pl.Config(tbl_rows=-1):
    # 1. Final labels per period
    print(result.filter(pl.col("label").is_not_null())
                .group_by("truth", "label").len()
                .with_columns((pl.col("len") / pl.col("len").sum().over("truth"))
                              .round(4).alias("share"))
                .sort("truth", "label"))

    # 2. Effect of smoothing per operating mode, in the known periods
    known = result.filter(pl.col("truth").is_in(["no_drift", "drift_1"])
                          & pl.col("label").is_not_null())
    print(known.group_by("operating_mode").agg(
        (pl.col("pred") == pl.col("truth")).mean().round(4).alias("raw_accuracy"),
        (pl.col("label") == pl.col("truth")).mean().round(4).alias("smoothed_accuracy"),
    ).sort("operating_mode"))

    # 3. Detection delay for the new drift, per machine
    first_alarm = (
        result.filter((pl.col("timestamp") >= NEW_DRIFT_START) & (pl.col("label") == "new_drift"))
              .group_by("machine_id").agg(pl.col("timestamp").min().alias("first_alarm"))
              .with_columns((pl.col("first_alarm") - pl.lit(NEW_DRIFT_START))
                            .dt.total_hours().alias("delay_hours"))
              .sort("machine_id")
    )
    print(first_alarm)

print("median delay (hours):", first_alarm["delay_hours"].median())
print("false new-drift rate:", known.select((pl.col("label") == "new_drift").mean()).item())
print("outlier rate in known periods:", known.select((pl.col("label") == "outlier").mean()).item())

shape: (5, 4)
┌───────────┬───────────┬───────┬────────┐
│ truth     ┆ label     ┆ len   ┆ share  │
│ ---       ┆ ---       ┆ ---   ┆ ---    │
│ str       ┆ str       ┆ u32   ┆ f64    │
╞═══════════╪═══════════╪═══════╪════════╡
│ drift_1   ┆ drift_1   ┆ 95338 ┆ 0.9992 │
│ drift_1   ┆ outlier   ┆ 78    ┆ 0.0008 │
│ new_drift ┆ drift_1   ┆ 788   ┆ 0.0101 │
│ new_drift ┆ new_drift ┆ 77119 ┆ 0.9858 │
│ new_drift ┆ outlier   ┆ 320   ┆ 0.0041 │
└───────────┴───────────┴───────┴────────┘
shape: (4, 3)
┌────────────────┬──────────────┬───────────────────┐
│ operating_mode ┆ raw_accuracy ┆ smoothed_accuracy │
│ ---            ┆ ---          ┆ ---               │
│ str            ┆ f64          ┆ f64               │
╞════════════════╪══════════════╪═══════════════════╡
│ HIGH_LOAD      ┆ 0.9994       ┆ 0.9994            │
│ IDLE           ┆ 0.8893       ┆ 0.9987            │
│ NORMAL         ┆ 0.9982       ┆ 0.9992            │
│ OVERLOAD       ┆ 0.9983       ┆ 0.9983            │
└────────────

In [9]:
LABEL_ORDER = ["no_drift", "drift_1", "outlier", "new_drift"]

def confusion(frame, truth_col="truth", pred_col="label"):
    m = (frame.filter(pl.col(pred_col).is_not_null())
              .group_by(truth_col, pred_col).len()
              .pivot(on=pred_col, index=truth_col, values="len")
              .fill_null(0)
              .sort(truth_col))
    return m.select(truth_col, *[c for c in LABEL_ORDER if c in m.columns])

print(confusion(result))

shape: (2, 4)
┌───────────┬─────────┬─────────┬───────────┐
│ truth     ┆ drift_1 ┆ outlier ┆ new_drift │
│ ---       ┆ ---     ┆ ---     ┆ ---       │
│ str       ┆ u32     ┆ u32     ┆ u32       │
╞═══════════╪═════════╪═════════╪═══════════╡
│ drift_1   ┆ 95338   ┆ 78      ┆ 0         │
│ new_drift ┆ 788     ┆ 320     ┆ 77119     │
└───────────┴─────────┴─────────┴───────────┘


In [10]:
det = result.filter(pl.col("label").is_not_null()).select(
    (pl.col("truth") == "new_drift").alias("is_new_drift"),
    (pl.col("label") == "new_drift").alias("alarm"),
)
print(det.group_by("is_new_drift", "alarm").len().sort("is_new_drift", "alarm"))

tp = det.filter(pl.col("is_new_drift") & pl.col("alarm")).height
fp = det.filter(~pl.col("is_new_drift") & pl.col("alarm")).height
fn = det.filter(pl.col("is_new_drift") & ~pl.col("alarm")).height
print(f"precision {tp / (tp + fp):.4f}, recall {tp / (tp + fn):.4f}")

shape: (3, 3)
┌──────────────┬───────┬───────┐
│ is_new_drift ┆ alarm ┆ len   │
│ ---          ┆ ---   ┆ ---   │
│ bool         ┆ bool  ┆ u32   │
╞══════════════╪═══════╪═══════╡
│ false        ┆ false ┆ 95416 │
│ true         ┆ false ┆ 1108  │
│ true         ┆ true  ┆ 77119 │
└──────────────┴───────┴───────┘
precision 1.0000, recall 0.9858


In [11]:
res_smoothed = apply_recent_rows_rule(res)
print(confusion(res_smoothed, truth_col="group"))

shape: (2, 4)
┌──────────┬──────────┬─────────┬─────────┐
│ group    ┆ no_drift ┆ drift_1 ┆ outlier │
│ ---      ┆ ---      ┆ ---     ┆ ---     │
│ str      ┆ u32      ┆ u32     ┆ u32     │
╞══════════╪══════════╪═════════╪═════════╡
│ drift_1  ┆ 220      ┆ 35073   ┆ 17      │
│ no_drift ┆ 42887    ┆ 0       ┆ 48      │
└──────────┴──────────┴─────────┴─────────┘
